In [15]:
from pathlib import Path
import sys
import pandas as pd

repo_root = Path.cwd().resolve().parent
if str(repo_root) not in sys.path:
    sys.path.append(str(repo_root))

from src.schema import (FaithfulnessRecord,
                        OriginalQuestion,
                        CounterfactualInfo, 
                        ReferenceModelInfo,
                        PredictorModelInfo,
                        MatchInfo,
                        CounterfactualDatabase)


### Adding original question and counterfactual
- Required fields are limited to:
    - question
    - dataset
    - question_idx
    - ground_truth
    - generator_model of counterfactual
    - generator_method of counterfactual
    - counterfactual

In [16]:
db = CounterfactualDatabase()

record1 = FaithfulnessRecord(
    original_question=OriginalQuestion(
        dataset="boolq",
        question="Is Paris the capital of France?",
        question_idx=118,
        ground_truth="Yes"
    ),
    counterfactual=CounterfactualInfo(
        generator_model="gpt-4",
        generator_method="replacement",
        question="Is Berlin the capital of France?"
    )
)

db.add_record(record1)

record2 = FaithfulnessRecord(
    original_question=OriginalQuestion(
        dataset="boolq",
        question="Is Paris the capital of France?",
        ground_truth="Yes",
        question_idx = 118,
    ),
    counterfactual=CounterfactualInfo(
        generator_model="gpt-4",
        generator_method="replacement",
        question="Is London the capital of France?"
    )
)

db.add_record(record2)


record3 = FaithfulnessRecord(
    original_question=OriginalQuestion(
        dataset="boolq",
        question="Is Paris the worst city in France?",
        ground_truth="Yes",
        question_idx = 120,
    ),
    counterfactual=CounterfactualInfo(
        generator_model="gpt-4",
        generator_method="replacement",
        question="Is Paris the best city in France?"
    )
)

db.add_record(record3)


## One example including all optional fields

In [17]:
record4 = FaithfulnessRecord(
    original_question=OriginalQuestion(
        dataset="BBQ",
        question="Is the sky blue?",
        ground_truth="Yes",
        question_idx=45,
    ),
    counterfactual=CounterfactualInfo(
        generator_model="gpt-5",
        generator_method="replacement",
        question="Is the sky green?",
        answer_first=True,
        ground_truth="No",
        coherence_scored_by_generator=True,
        coherence_explanation_by_generator="The question makes sense.",
        coherence_external_scoring_model="gpt-4",
        coherence_scored_by_external_model=True,
        coherence_explanation_by_external_model="The question makes sense.",
        hamming_distance=1,
    ),
    reference_model=ReferenceModelInfo(
        model="gemini-2.0-flash",
        answer="Yes",
        explanation="This is obvious and I'm brilliant",
        explanation_confidence="high"
    ),
    predictor_model=PredictorModelInfo(
        model="gpt-4",
        answer_with_explanation="Yes",
        answer_without_explanation="Yes",
        justification_with_explanation="insert a justification",
        justification_without_explanation="insert a justification",
        justification_confidence_with_explanation="medium",
        justification_confidence_without_explanation="low"
        ),
    match_info=MatchInfo(
        match_with_explanation=1,
        match_without_explanation=1,
        match_delta=0)
)

db.add_record(record4)

In [18]:
db.records

[FaithfulnessRecord(original_question=OriginalQuestion(dataset='boolq', question='Is Paris the capital of France?', question_idx=118, ground_truth='Yes'), counterfactual=CounterfactualInfo(generator_model='gpt-4', generator_method='replacement', question='Is Berlin the capital of France?', question_idx=0, answer_first=None, ground_truth=None, coherence_scored_by_generator=None, coherence_explanation_by_generator=None, coherence_external_scoring_model=None, coherence_scored_by_external_model=None, coherence_explanation_by_external_model=None, hamming_distance=None), reference_model=None, predictor_model=None, match_info=None),
 FaithfulnessRecord(original_question=OriginalQuestion(dataset='boolq', question='Is Paris the capital of France?', question_idx=118, ground_truth='Yes'), counterfactual=CounterfactualInfo(generator_model='gpt-4', generator_method='replacement', question='Is London the capital of France?', question_idx=1, answer_first=None, ground_truth=None, coherence_scored_by_g

### Save as parquet file

In [19]:
db.save_parquet("counterfactuals_test.parquet")

### Load parquet file as df

In [20]:
df = pd.read_parquet("counterfactuals_test.parquet")
df

,original_dataset,original_question,original_question_idx,original_ground_truth,counterfactual_generator_model,counterfactual_generator_method,counterfactual_question,counterfactual_question_idx,counterfactual_answer_first,counterfactual_ground_truth,...,predictor_model_model,predictor_model_answer_with_explanation,predictor_model_answer_without_explanation,predictor_model_justification_with_explanation,predictor_model_justification_without_explanation,predictor_model_justification_confidence_with_explanation,predictor_model_justification_confidence_without_explanation,match_match_with_explanation,match_match_without_explanation,match_match_delta
0,boolq,Is Paris the capital of France?,118,Yes,gpt-4,replacement,Is Berlin the capital of France?,0,None,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN
1,boolq,Is Paris the capital of France?,118,Yes,gpt-4,replacement,Is London the capital of France?,1,None,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN
2,boolq,Is Paris the worst city in France?,120,Yes,gpt-4,replacement,Is Paris the best city in France?,0,None,None,...,None,None,None,None,None,None,None,NaN,NaN,NaN
3,BBQ,Is the sky blue?,45,Yes,gpt-5,replacement,Is the sky green?,0,True,No,...,gpt-4,Yes,Yes,insert a justification,insert a justification,medium,low,1.0,1.0,0.0


### Load parquet file back into Database dataclass

In [21]:
db2 = CounterfactualDatabase.load_parquet("counterfactuals_test.parquet")

In [22]:
db2.records

[FaithfulnessRecord(original_question=OriginalQuestion(dataset='boolq', question='Is Paris the capital of France?', question_idx=118, ground_truth='Yes'), counterfactual=CounterfactualInfo(generator_model='gpt-4', generator_method='replacement', question='Is Berlin the capital of France?', question_idx=0, answer_first=None, ground_truth=None, coherence_scored_by_generator=None, coherence_explanation_by_generator=None, coherence_external_scoring_model=None, coherence_scored_by_external_model=None, coherence_explanation_by_external_model=None, hamming_distance=nan), reference_model=ReferenceModelInfo(model=None, answer=None, explanation=None, explanation_confidence=None), predictor_model=PredictorModelInfo(model=None, answer_with_explanation=None, answer_without_explanation=None, justification_with_explanation=None, justification_without_explanation=None, justification_confidence_with_explanation=None, justification_confidence_without_explanation=None), match_info=MatchInfo(match_with_ex